# Libraries

In [1]:
import configparser
import json
import os
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import requests
from dotenv import load_dotenv
from rdflib import RDF, RDFS, Graph, Literal, URIRef
from rdflib.namespace import XSD
from urllib3.exceptions import InsecureRequestWarning

# Parsing inputs

In [2]:
def load_config(config_path: Path) -> dict[str, dict[str, str]]:
    config_path = Path(config_path).resolve()
    if not config_path.is_file():
        raise FileNotFoundError(f"File not found: {config_path}")

    config = configparser.ConfigParser()
    config.read(config_path)

    data = {section: dict(config.items(section)) for section in config.sections()}

    return data

In [3]:
def load_pipeline(config_path: Path) -> dict[str, dict[str, str]]:
    # 2 functions one for config_ini and one for pipeline_ini
    # put default values when not found or rise error
    # TO IMPLEMENT
    config_path = Path(config_path).resolve()
    if not config_path.is_file():
        raise FileNotFoundError(f"File not found: {config_path}")

    config = configparser.ConfigParser()
    config.read(config_path)

    data = {section: dict(config.items(section)) for section in config.sections()}

    return data

In [4]:
def parse_bool(value: str | bool | None, default: bool) -> bool:
    if value is None:
        return default

    # already boolean
    if isinstance(value, bool):
        return value

    s = str(value).strip()

    # common boolean forms
    if s.lower() in {"false", "0", "no", "off", "False", "FALSE", "F"}:
        return False
    if s.lower() in {"true", "1", "yes", "on", "True", "TRUE", "T"}:
        return True

    # unknown value, return default
    return default

# RML mapping

In [5]:
def rml_execute(config_path: str) -> Path:
    config = load_config(config_path)
    config_path = Path(config_path).resolve()

    output_value = None
    output_section = None
    for section, kv in config.items():
        if "output_file" in kv:
            output_value = kv["output_file"]
            output_section = section
            break

    if output_value is None:
        raise ValueError(
            "No 'output_file' defined in the config file (in any section). "
            "Add something like:\n\n"
            "output_file = output.nt\n"
        )

    output_path = Path(output_value)
    if not output_path.is_absolute():
        output_path = (config_path.parent / output_path).resolve()
    else:
        output_path = output_path.resolve()

    cmd = [sys.executable, "-m", "morph_kgc", str(config_path)]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            "Morph-KGC execution failed.\n\n"
            f"Config: {config_path}\n"
            f"Detected output_file in section [{output_section}]: {output_value}\n\n"
            f"STDERR:\n{result.stderr}\n\n"
            f"STDOUT:\n{result.stdout}"
        )

    if not output_path.exists():
        raise FileNotFoundError(
            "Morph-KGC finished without error, but output file was not found.\n\n"
            f"Expected: {output_path}\n"
            f"Config: {config_path}\n"
            f"Detected output_file in section [{output_section}]: {output_value}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return output_path

# Wikibase api

In [6]:
class WikibaseAPI:
    def __init__(self, params: dict) -> None:
        # ADD LATER IF SPARQL -> SIMPLER
        # ADD LATER IF SEARCH WIDE CAN BE GIVEN TO THE PIPELINE.INI
        self.api_url = params["api_url"]
        if not self.api_url:
            raise ValueError(
                "Wikibase API URL cannot be empty."
                "Add something like:\n\n"
                "api_url = https://wikibase.be/w/api.php\n"
            )
        self.session = requests.Session()
        self.language = params.get("language", "en")
        self.create_prop = params["create_missing_properties"]
        self.verify = params["tls_verify"]
        if self.verify is False:
            warnings.simplefilter("ignore", InsecureRequestWarning)
        self.user, self.password = self._load_env()
        self._login()

    def _load_env(self) -> tuple[str, str]:
        load_dotenv()
        user = os.getenv("WB_USER")
        password = os.getenv("WB_PASSWORD")
        if not user or not password:
            raise OSError(
                "\nWikibase credentials not found.\n\n"
                "Expected a .env file at the project root with:\n\n"
                "WB_USER=your_username\n"
                "WB_PASSWORD=your_password\n\n"
            )
        return user, password

    def _get_token(self, token_type: str) -> str:
        r = self.session.get(
            self.api_url,
            params={
                "action": "query",
                "meta": "tokens",
                "type": token_type,
                "format": "json",
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        return r.json()["query"]["tokens"][f"{token_type}token"]

    def _login(self) -> None:
        login_token = self._get_token("login")

        r = self.session.post(
            self.api_url,
            data={
                "action": "login",
                "lgname": self.user,
                "lgpassword": self.password,
                "lgtoken": login_token,
                "format": "json",
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        data = r.json()

        result = data.get("login", {}).get("result")
        if result != "Success":
            raise ValueError(f"Login failed: {data}")

        self.csrf_token = self._get_token("csrf")

    def search_item(
        self, text: str, language: str = "en", limit: int = 1
    ) -> list[dict]:
        r = self.session.get(
            self.api_url,
            params={
                "action": "wbsearchentities",
                "format": "json",
                "search": text,
                "language": language,
                "type": "item",
                "limit": limit,
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        return r.json().get("search", [])

    def search_property(
        self,
        text: str,
        language: str = "en",
        datatype: str = None,
        search_wide: int = 10,
        limit: int = 1,
    ) -> list[dict]:
        """
        Search for properties and optionally filter by datatype
        """
        # Step 1: Search for properties
        r = self.session.get(
            self.api_url,
            params={
                "action": "wbsearchentities",
                "format": "json",
                "search": text,
                "language": language,
                "type": "property",
                "limit": search_wide,
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        results = r.json().get("search", [])

        if not results:
            return []

        # Step 2: Filter by datatype if needed
        if datatype:
            # Get datatypes for all search results at once
            pids = [res["id"] for res in results]
            datatypes = self.get_properties_datatypes(pids)

            # Filter results by datatype
            filtered_results = []
            for res in results:
                if datatypes.get(res["id"]) == datatype:
                    # Add datatype to the result for caching
                    res["datatype"] = datatype
                    filtered_results.append(res)

            results = filtered_results

        # Step 3: Apply limit
        if len(results) > limit:
            results = results[:limit]

        return results

    def get_properties_datatypes(self, pids: list[str]) -> dict[str, str]:
        """
        Get datatypes for multiple property IDs at once
        """
        if not pids:
            return {}

        # Remove duplicates
        pids = list(set(pids))

        # Wikibase API accepts pipe-separated IDs, but limit may apply
        # Split into chunks if needed (some wikibase instances have limits)
        chunk_size = 50  # Safe limit for most wikibase instances
        all_datatypes = {}

        for i in range(0, len(pids), chunk_size):
            chunk = pids[i : i + chunk_size]

            r = self.session.get(
                self.api_url,
                params={
                    "action": "wbgetentities",
                    "format": "json",
                    "ids": "|".join(chunk),
                    "props": "datatype",  # Only request datatype to save bandwidth
                },
                timeout=30,
                verify=self.verify,
            )
            r.raise_for_status()
            data = r.json()

            # Extract datatypes
            for pid in chunk:
                entity = data.get("entities", {}).get(pid)
                if entity:
                    all_datatypes[pid] = entity.get("datatype")

        return all_datatypes

    def create_item(self, label: str, language: str = "en") -> str:
        """
        Creates a Wikibase item with a label (and optional description). Returns Q-id.
        """
        payload = {
            "labels": {language: {"language": language, "value": label}},
        }

        r = self.session.post(
            self.api_url,
            data={
                "action": "wbeditentity",
                "format": "json",
                "new": "item",
                "data": json.dumps(payload),
                "token": self.csrf_token,
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        data = r.json()
        try:
            return data["entity"]["id"]
        except Exception:
            raise RuntimeError(f"Item creation failed: {data}")

    def create_property(self, label: str, datatype: str, language: str = "en") -> str:
        """
        Creates a Wikibase property. Returns P-id.
        NOTE: requires rights + correct datatype choice.
        """
        payload = {
            "labels": {language: {"language": language, "value": label}},
            "datatype": datatype,
        }

        r = self.session.post(
            self.api_url,
            data={
                "action": "wbeditentity",
                "format": "json",
                "new": "property",
                "data": json.dumps(payload),
                "token": self.csrf_token,
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        data = r.json()
        try:
            return data["entity"]["id"]
        except Exception:
            raise RuntimeError(f"Property creation failed: {data}")

    def add_naive_claim(self, subject_qid: str, pid: str, datavalue: dict) -> dict:
        # Naive = cretes a new claim without checking if it already exists, and without qualifiers or references.
        claim = {
            "mainsnak": {
                "snaktype": "value",
                "property": pid,
                "datavalue": datavalue,
            },
            "type": "statement",
            "rank": "normal",
        }

        payload = {"claims": {pid: [claim]}}

        r = self.session.post(
            self.api_url,
            data={
                "action": "wbeditentity",
                "format": "json",
                "id": subject_qid,
                "data": json.dumps(payload),
                "token": self.csrf_token,
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        return r.json()

    def get_item_details(self, item_id: str) -> dict:
        try:
            r = self.session.get(
                self.api_url,
                params={
                    "action": "wbgetentities",
                    "format": "json",
                    "ids": item_id,
                    "props": "labels|aliases",  # Request only what we need
                },
                timeout=30,
                verify=self.verify,
            )
            r.raise_for_status()
            data = r.json()

            # Return the entity data or empty dict
            return data.get("entities", {}).get(item_id, {})

        except Exception as e:
            print(f"Error getting item details for {item_id}: {e}")
            return {}

    def add_item_alias(self, item_id: str, alias: str, language: str = "en") -> dict:
        data = {"aliases": {language: [alias]}}

        try:
            r = self.session.post(
                self.api_url,
                data={
                    "action": "wbeditentity",
                    "format": "json",
                    "id": item_id,
                    "data": json.dumps(data),
                    "token": self.csrf_token,
                },
                timeout=30,
                verify=self.verify,
            )
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"Error adding alias to {item_id}: {e}")
            return {"error": {"info": str(e)}}

    def update_item_label(self, item_id: str, label: str, language: str = "en") -> dict:
        data = {"labels": {language: {"language": language, "value": label}}}

        try:
            r = self.session.post(
                self.api_url,
                data={
                    "action": "wbeditentity",
                    "format": "json",
                    "id": item_id,
                    "data": json.dumps(data),
                    "token": self.csrf_token,
                },
                timeout=30,
                verify=self.verify,
            )
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"Error updating label for {item_id}: {e}")
            return {"error": {"info": str(e)}}

    # function for testing purposes, not used in the pipeline
    def delete(self, entity_id: str) -> dict:
        """
        Delete an entity by id: Qxxx or Pxxx.
        Requires user rights to delete pages/entities.
        Returns the JSON response.
        """
        if entity_id.startswith("Q"):
            title = f"Item:{entity_id}"
        elif entity_id.startswith("P"):
            title = f"Property:{entity_id}"
        else:
            raise ValueError("entity_id must start with 'Q' or 'P' (e.g., Q42, P17).")

        r = self.session.post(
            self.api_url,
            data={
                "action": "delete",
                "title": title,
                "token": self.csrf_token,
                "format": "json",
            },
            timeout=30,
            verify=self.verify,
        )
        r.raise_for_status()
        return r.json()

# Namespaces

In [7]:
@dataclass(frozen=True)
class Prefixes:
    q: str
    p: str

In [8]:
def prefixes(q: str | None = None, p: str | None = None) -> Prefixes:
    qv = (q or "urn:wikibase:Q:").strip()
    pv = (p or "urn:wikibase:P:").strip()

    if not qv or not pv:
        raise ValueError("Prefixes cannot be empty.")
    if qv == pv:
        raise ValueError("Item and property namespaces cannot be the same.")
    return Prefixes(q=qv, p=pv)

In [11]:
def urn_suffix(urn: str, prefix: str) -> str:
    """Extract the suffix part of a URN by removing the prefix."""
    if urn.startswith(prefix):
        return urn[len(prefix) :]
    return urn

# Lookup cache

In [12]:
def load_lookup(path: str | Path) -> dict:
    path = Path(path)
    if not path.exists():
        return {"items": {}, "properties": {}}
    data = json.loads(path.read_text(encoding="utf-8"))
    data.setdefault("items", {})
    data.setdefault("properties", {})
    return data

In [13]:
def save_lookup(path: str | Path, lookup: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(lookup, indent=2, ensure_ascii=False), encoding="utf-8")

# RDF parsing

In [14]:
def read_nt_triples(nt_path: Path) -> Graph:
    g = Graph()
    g.parse(str(nt_path), format="nt")
    return g

In [15]:
def extract_labels_from_nt(
    g: Graph | Path, item_prefix: str, property_prefix: str, lang: str = "en"
) -> tuple[dict[str, str], dict[str, str]]:
    if isinstance(g, Path):
        graph = Graph()
        graph.parse(g, format="nt")
    else:
        graph = g

    item_labels = {}
    property_labels = {}

    for s, p, o in graph.triples((None, RDFS.label, None)):
        if not (isinstance(o, Literal) and o.language == lang):
            continue

        s_str = str(s)
        if s_str.startswith(item_prefix):
            item_labels[s_str] = str(o)
        elif s_str.startswith(property_prefix):
            property_labels[s_str] = str(o)

    return item_labels, property_labels

In [16]:
def urn_suffix(urn: str, prefix: str) -> str:
    if urn.startswith(prefix):
        return urn[len(prefix) :]
    return urn

In [17]:
def determine_datatype(o_term, prefix: Prefixes) -> str:
    # Case 1: URI Reference (could be internal or external)
    if isinstance(o_term, URIRef):
        o_str = str(o_term)
        if o_str.startswith(prefix.q):
            return "wikibase-item"  # Internal item reference
        else:
            return "url"  # External URI

    # Case 2: Literal (typed or language-tagged)
    elif isinstance(o_term, Literal):
        # Check for language tag first
        if o_term.language:
            return "monolingualtext"

        # Check datatype
        if o_term.datatype:
            # Numeric types
            if o_term.datatype in [
                XSD.integer,
                XSD.int,
                XSD.long,
                XSD.short,
                XSD.byte,
                XSD.unsignedLong,
                XSD.unsignedInt,
                XSD.unsignedShort,
                XSD.unsignedByte,
                XSD.positiveInteger,
                XSD.negativeInteger,
                XSD.nonPositiveInteger,
                XSD.nonNegativeInteger,
                XSD.decimal,
                XSD.float,
                XSD.double,
            ]:
                return "quantity"

            # Boolean
            elif o_term.datatype == XSD.boolean:
                return "boolean"

            # Date/Time types
            elif o_term.datatype in [
                XSD.date,
                XSD.dateTime,
                XSD.time,
                XSD.gYear,
                XSD.gYearMonth,
                XSD.gMonth,
                XSD.gMonthDay,
                XSD.gDay,
            ]:
                return "time"

            # URI types
            elif o_term.datatype in [XSD.anyURI, XSD.NCName, XSD.Name, XSD.token]:
                return "url"

            # String types
            elif o_term.datatype in [
                XSD.string,
                XSD.normalizedString,
                XSD.language,
                XSD.NMTOKEN,
                XSD.NMTOKENS,
            ]:
                return "string"

            # Unknown datatype - default to string
            else:
                print(
                    f"WARNING: Unknown datatype {o_term.datatype}, defaulting to string"
                )
                return "string"
        else:
            # Plain literal without datatype
            return "string"

    # Case 3: Blank node or other
    else:
        print(f"WARNING: Unexpected term type {type(o_term)}, defaulting to string")
        return "string"

In [18]:
def create_datavalue(o_term, datatype: str, prefix: Prefixes, o_id: str = None) -> dict:
    if datatype == "wikibase-item":
        if not o_id:
            raise ValueError("o_id is required for wikibase-item datatype")
        return {
            "value": {"entity-type": "item", "id": o_id},
            "type": "wikibase-entityid",
        }

    elif datatype == "url":
        # Handle URL/URI
        if isinstance(o_term, Literal):
            value = str(o_term)
        else:
            value = str(o_term)
        return {"value": value, "type": "string"}

    elif datatype == "quantity":
        # Handle quantity values
        if isinstance(o_term, Literal):
            try:
                # Convert to appropriate numeric type
                if o_term.datatype in [XSD.integer, XSD.int, XSD.long, XSD.short]:
                    value = int(o_term)
                else:
                    value = float(o_term)

                return {
                    "value": {"amount": str(value), "unit": "1"},
                    "type": "quantity",
                }
            except (ValueError, TypeError):
                # If conversion fails, treat as string
                print(
                    f"WARNING: Could not parse '{o_term}' as quantity, treating as string"
                )
                return {"value": str(o_term), "type": "string"}
        else:
            # Not a literal, treat as string
            return {"value": str(o_term), "type": "string"}

    elif datatype == "boolean":
        # Handle boolean values
        if isinstance(o_term, Literal):
            if o_term.datatype == XSD.boolean:
                value = o_term.value
            else:
                # Try to interpret string as boolean
                value = str(o_term).lower() in {"true", "1", "yes", "on", "t", "y"}
        else:
            value = False
        return {"value": value, "type": "boolean"}

    elif datatype == "time":
        # Handle date/time values
        if isinstance(o_term, Literal):
            # Extract the ISO date string
            value = str(o_term)

            # Format for Wikibase
            if value and not value.startswith("+"):
                value = f"+{value}"
            if "T" not in value and len(value) > 5:
                value = f"{value}T00:00:00Z"

            # Determine precision based on string length
            if len(value) <= 5:  # Just year
                precision = 9
            elif len(value) <= 8:  # Year-month
                precision = 10
            else:  # Full date
                precision = 11

            return {
                "value": {
                    "time": value,
                    "precision": precision,
                    "calendarmodel": "http://www.wikidata.org/entity/Q1985727",
                },
                "type": "time",
            }
        else:
            print(f"WARNING: Expected literal for time, got {type(o_term)}")
            return {"value": str(o_term), "type": "string"}

    elif datatype == "monolingualtext":
        # Handle language-tagged strings
        if isinstance(o_term, Literal) and o_term.language:
            return {
                "value": {"text": str(o_term), "language": o_term.language},
                "type": "monolingualtext",
            }
        else:
            # Not a language-tagged literal, treat as string
            print(f"WARNING: Expected language-tagged literal, got: {o_term}")
            return {"value": str(o_term), "type": "string"}

    else:  # Default to string
        return {"value": str(o_term), "type": "string"}

# Wikibase populator

## Process p

In [19]:
PREDICATE_HANDLERS = {
    RDF.type: "TYPE",  # This is a URIRef object
    RDFS.label: "LABEL",  # This is a URIRef object
}


def process_predicate(
    wikibase_api: WikibaseAPI,
    p_str: str,
    property_labels: dict[str, str],
    prefix: Prefixes,
    lookup_cache: dict,
    datatype: str,
) -> tuple[str | None, dict]:
    # Convert string to URIRef for comparison
    p_uriref = URIRef(p_str)

    # Regular property in your namespace
    if p_str.startswith(prefix.p):
        if p_str in lookup_cache["properties"]:
            p_id = lookup_cache["properties"][p_str]
        else:
            label = property_labels.get(p_str)
            if not label:
                label = urn_suffix(p_str, prefix.p)

            # Search for existing property
            search_result = wikibase_api.search_property(
                label,
                language=wikibase_api.language,
                datatype=datatype,
                search_wide=20,
                limit=1,
            )

            if search_result:
                p_id = search_result[0]["id"]
                print(f"Found existing property '{label}' ({datatype}): {p_id}")
            else:
                if not wikibase_api.create_prop:
                    print(
                        f"Property '{label}' not found and creation is disabled. Skipping predicate '{p_str}'."
                    )
                    return None, lookup_cache
                p_id = wikibase_api.create_property(
                    label, datatype=datatype, language=wikibase_api.language
                )
                print(f"Created new property '{label}' ({datatype}): {p_id}")

            lookup_cache["properties"][p_str] = p_id
        return p_id, lookup_cache

    # Check if this is a handled predicate (using URIRef, not string)
    handler = PREDICATE_HANDLERS.get(p_uriref)  # Use URIRef as key!
    if handler:
        print(f"Detected special predicate: {handler} -> {p_str}")
        # Initialize handlers dict in lookup_cache if needed
        if "handlers" not in lookup_cache:
            lookup_cache["handlers"] = {}
        lookup_cache["handlers"][p_str] = handler
        return handler, lookup_cache

    # Unknown predicate
    else:
        print(
            f"Skipping predicate outside property namespace and unknown prefix: {p_str}"
        )
        return None, lookup_cache

In [20]:
def handle_type(
    wikibase_api: WikibaseAPI, s_id: str, o_id: str, instance_of_property: str = "P31"
) -> dict:
    if not o_id:
        error_msg = "Cannot handle rdf:type claim without a valid object ID"
        print(f"ERROR: {error_msg}")
        return {"error": error_msg}

    # Create datavalue for the type item
    datavalue = {
        "value": {"entity-type": "item", "id": o_id},
        "type": "wikibase-entityid",
    }

    # Add the claim
    result = wikibase_api.add_naive_claim(s_id, instance_of_property, datavalue)

    if "error" not in result:
        print(f"Added type: {s_id} -> {o_id} (as {instance_of_property})")
    else:
        print(
            f"Failed to add type: {result.get('error', {}).get('info', 'Unknown error')}"
        )

    return result

In [21]:
def handle_label(
    wikibase_api: WikibaseAPI, s_id: str, o_term, prefix: Prefixes
) -> dict:
    # Check if it's a language-tagged literal
    if not isinstance(o_term, Literal) or not o_term.language:
        print(f"WARNING: Expected language-tagged literal for label, got: {o_term}")
        return {"error": "Not a language-tagged literal"}

    value = str(o_term)
    lang = o_term.language

    # Get current item details to check existing label
    item_details = wikibase_api.get_item_details(s_id)

    if item_details and "labels" in item_details:
        current_label = item_details["labels"].get(lang, {}).get("value", "")

        if current_label and current_label != value:
            # Different from current label -> add as alias
            print(f"Adding alias for {s_id}: '{value}' (@{lang})")
            return wikibase_api.add_item_alias(s_id, value, lang)
        elif not current_label:
            # No label in this language -> set as label
            print(f"Setting label for {s_id}: '{value}' (@{lang})")
            return wikibase_api.update_item_label(s_id, value, lang)
        else:
            # Same as current label -> skip
            print(f"Label '{value}' already exists for {s_id}, skipping")
            return {"success": True, "message": "Label already exists"}
    else:
        # No labels found, set as label
        print(f"Setting label for {s_id}: '{value}' (@{lang})")
        return wikibase_api.update_item_label(s_id, value, lang)

## Process s and o

In [22]:
def process_item(
    wikibase_api: WikibaseAPI,
    item_str: str,
    item_labels: dict,
    prefix: Prefixes,
    lookup_cache: dict,
) -> tuple[str | None, dict]:
    # Check cache first
    if item_str in lookup_cache["items"]:
        return lookup_cache["items"][item_str], lookup_cache

    # Get label for the item
    label = item_labels.get(item_str)
    if not label:
        label = urn_suffix(item_str, prefix.q)

    # Search for existing item
    try:
        search_result = wikibase_api.search_item(
            label, language=wikibase_api.language, limit=1
        )

        if search_result:
            item_id = search_result[0]["id"]
            print(f"Found existing item '{label}': {item_id}")
        else:
            # Check if item creation is enabled
            if not wikibase_api.create_prop:
                print(
                    f"Item '{label}' not found and creation is disabled. Skipping {item_str}."
                )
                return None, lookup_cache

            # Create new item
            item_id = wikibase_api.create_item(label, language=wikibase_api.language)
            print(f"Created new item '{label}': {item_id}")

        # Cache the item ID
        lookup_cache["items"][item_str] = item_id
        return item_id, lookup_cache

    except Exception as e:
        print(f"Error processing item {item_str}: {e}")
        return None, lookup_cache

## Main populator function

In [23]:
def populator(
    wikibase_api: WikibaseAPI, rdf_path: Path, prefix: Prefixes, lookup_cache: dict
) -> None:
    g = read_nt_triples(rdf_path)

    item_labels, property_labels = extract_labels_from_nt(
        g, prefix.q, prefix.p, lang=wikibase_api.language
    )

    stats = {"success": 0, "skipped": 0, "errors": 0}

    for s, p, o in g:
        s_str = str(s)
        p_str = str(p)

        try:
            # Process predicate first to know what we're dealing with
            datatype = determine_datatype(o, prefix)
            p_result, lookup_cache = process_predicate(
                wikibase_api, p_str, property_labels, prefix, lookup_cache, datatype
            )

            if not p_result:
                print(f"Skipping triple with unhandled predicate: {p_str}")
                stats["skipped"] += 1
                continue

            # Process subject - CHECK FOR NONE RETURN
            if not s_str.startswith(prefix.q):
                print(f"Skipping triple with subject outside item namespace: {s_str}")
                stats["skipped"] += 1
                continue

            s_result = process_item(
                wikibase_api, s_str, item_labels, prefix, lookup_cache
            )
            if s_result[0] is None:  # Unpack tuple and check first value
                print(f"Failed to process subject {s_str}, skipping triple")
                stats["errors"] += 1
                continue
            s_id, lookup_cache = s_result

            # Process object (if it's an item URI)
            o_id = None
            if isinstance(o, URIRef) and str(o).startswith(prefix.q):
                o_result = process_item(
                    wikibase_api, str(o), item_labels, prefix, lookup_cache
                )
                if o_result[0] is None:
                    print(f"Failed to process object {o}, skipping triple")
                    stats["errors"] += 1
                    continue
                o_id, lookup_cache = o_result

            # Handle based on predicate type
            if p_result == "TYPE":
                result = handle_type(wikibase_api, s_id, o_id)

            elif p_result == "LABEL":
                result = handle_label(wikibase_api, s_id, o, prefix)

            else:  # Regular property
                p_id = p_result
                datavalue = create_datavalue(o, datatype, prefix, o_id)
                result = wikibase_api.add_naive_claim(s_id, p_id, datavalue)

            # Check result
            if "error" not in result:
                print(f"Successfully processed: {s_id} {p_str} -> {str(o)[:50]}...")
                stats["success"] += 1
            else:
                print(f"Failed: {result.get('error', {}).get('info', 'Unknown error')}")
                stats["errors"] += 1

        except Exception as e:
            print(
                f"Unexpected error processing triple {s_str} {p_str} {str(o)[:50]}: {e}"
            )
            stats["errors"] += 1
            continue

    # Print summary
    print("\n" + "=" * 50)
    print("POPULATOR SUMMARY")
    print("=" * 50)
    print(f"Successfully processed: {stats['success']}")
    print(f"Skipped: {stats['skipped']}")
    print(f"Errors: {stats['errors']}")
    print(f"Total triples: {stats['success'] + stats['skipped'] + stats['errors']}")

    return lookup_cache

# Global function

In [24]:
def update(config_ini: str, pipeline_ini: str) -> None:
    output_path = rml_execute(config_ini)
    pipeline = load_config(pipeline_ini)

    pipeline["wikibase"]["tls_verify"] = parse_bool(
        pipeline["wikibase"]["tls_verify"], default=True
    )
    pipeline["wikibase"]["create_missing_properties"] = parse_bool(
        pipeline["wikibase"]["create_missing_properties"], default=False
    )
    pipeline["cache"]["store_file"] = parse_bool(
        pipeline["cache"]["store_file"], default=False
    )

    wb_api = WikibaseAPI(pipeline["wikibase"])
    prefix = prefixes(
        pipeline["urn"]["item_prefix"], pipeline["urn"]["property_prefix"]
    )
    lookup_cache = load_lookup(pipeline["cache"]["lookup_file"])

    lookup_cache = populator(wb_api, output_path, prefix, lookup_cache)

    if pipeline["cache"]["store_file"]:
        save_lookup(pipeline["cache"]["lookup_file"], lookup_cache)

# Tests

In [29]:
pipeline = load_config("../config/pipeline.ini")

In [30]:
output_path = rml_execute("../config/config.ini")

RuntimeError: Morph-KGC execution failed.

Config: C:\Users\liu00\Documents\Université\Master2\TFE\ATFE9009-wikibase-rdf-pipeline\config\config.ini
Detected output_file in section [CONFIGURATION]: ..//data/output/out.nt

STDERR:
c:\Users\liu00\Documents\UniversitÃ©\Master2\TFE\ATFE9009-wikibase-rdf-pipeline\.venv\Scripts\python.exe: No module named morph_kgc


STDOUT:


In [45]:
output_path = Path("out.nt")

In [ ]:
pipeline["wikibase"]["tls_verify"] = parse_bool(
    pipeline["wikibase"]["tls_verify"], default=True
)
pipeline["wikibase"]["create_missing_properties"] = parse_bool(
    pipeline["wikibase"]["create_missing_properties"], default=False
)
pipeline["cache"]["store_file"] = parse_bool(
    pipeline["cache"]["store_file"], default=False
)

In [ ]:
wb_api = WikibaseAPI(pipeline["wikibase"])
prefix = prefixes(pipeline["urn"]["item_prefix"], pipeline["urn"]["property_prefix"])
lookup_cache = load_lookup(pipeline["cache"]["lookup_file"])

In [ ]:
populator(wb_api, output_path, prefix, lookup_cache)

Found existing property 'name' (string): P1
Found existing item 'Pikachu': Q118
Successfully processed: Q118 urn:wikibase:P:name -> Pikachu...
Property 'pokedexNumber' not found and creation is disabled. Skipping predicate 'urn:wikibase:P:pokedexNumber'.
Skipping triple with unhandled predicate: urn:wikibase:P:pokedexNumber
Property 'attack' not found and creation is disabled. Skipping predicate 'urn:wikibase:P:attack'.
Skipping triple with unhandled predicate: urn:wikibase:P:attack
Property 'gender' not found and creation is disabled. Skipping predicate 'urn:wikibase:P:gender'.
Skipping triple with unhandled predicate: urn:wikibase:P:gender
Property 'specialDefense' not found and creation is disabled. Skipping predicate 'urn:wikibase:P:specialDefense'.
Skipping triple with unhandled predicate: urn:wikibase:P:specialDefense
Property 'eggGroup' not found and creation is disabled. Skipping predicate 'urn:wikibase:P:eggGroup'.
Skipping triple with unhandled predicate: urn:wikibase:P:eggGr